In [246]:
import pandas as pd

In [247]:
df = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json")

In [248]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38966 entries, 0 to 38965
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        38966 non-null  object
 1   카테고리       38966 non-null  object
 2   대화셋일련번호    38966 non-null  object
 3   화자         38966 non-null  object
 4   문장번호       38966 non-null  int64 
 5   고객의도       38966 non-null  object
 6   상담사의도      38966 non-null  object
 7   QA         38966 non-null  object
 8   고객질문(요청)   38966 non-null  object
 9   상담사질문(요청)  38966 non-null  object
 10  고객답변       38966 non-null  object
 11  상담사답변      38966 non-null  object
 12  개체명        38966 non-null  object
 13  용어사전       38966 non-null  object
 14  지식베이스      38966 non-null  object
dtypes: int64(1), object(14)
memory usage: 4.5+ MB


In [249]:
# 결측치의 개수를 확인
df.isna().sum()

도메인          0
카테고리         0
대화셋일련번호      0
화자           0
문장번호         0
고객의도         0
상담사의도        0
QA           0
고객질문(요청)     0
상담사질문(요청)    0
고객답변         0
상담사답변        0
개체명          0
용어사전         0
지식베이스        0
dtype: int64

In [250]:
df.head()

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,1,버스노선,,Q,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,,,,"서울, 가산동, 남대문시장, 버스, 노선",서울/지명/ 가산동/동네/ 남대문시장/지명/ 버스/교통수단,"가산동,교통수단"
1,다산콜센터,대중교통 안내,B2033,상담사,2,,버스노선,Q,,가산동 어디에서 출발하십니까?,,,"가산동, 출발",가산동/동네/ 출발/출발지,"출발,출발지"
2,다산콜센터,대중교통 안내,B2033,고객,3,버스노선,,A,,,가산동 주민센터입니다.,,"가산동, 주민센터",가산동/동네/ 주민센터/공공기관,"주민센터,공공기관"
3,다산콜센터,대중교통 안내,B2033,상담사,4,,버스노선,A,,,,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.,"가산동, 주민센터, 남대문시장,버스, 노선",가산동/동네/ 주민센터/공공기관/ 남대문시장/지명/ 버스/교통수단,"주민센터,교통수단"
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,,정류장,,정류장


### 문제
1. 일반행정 데이터와, 대중 교통 데이터를 로드
2. 두개의 데이터프레임을 결합 (단순한 행 결합)
3. 데이터의 필터링 고객질문에 대한 상담사의 답변이 즉각적으로 오는 데이터들만 필터
4. 질문 중 중복 데이터를 제거
5. 질문들을 모아서 토큰화, 벡터화
6. 그 외의 질문 목록을 이용하여 코사인 유사도 확인하고 유사 질문과 답변을 출력

In [251]:
df1 = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json")

In [252]:
df_all = pd.concat((df, df1), ignore_index=True)

In [253]:
df_all.head(6)

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,1,버스노선,,Q,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,,,,"서울, 가산동, 남대문시장, 버스, 노선",서울/지명/ 가산동/동네/ 남대문시장/지명/ 버스/교통수단,"가산동,교통수단"
1,다산콜센터,대중교통 안내,B2033,상담사,2,,버스노선,Q,,가산동 어디에서 출발하십니까?,,,"가산동, 출발",가산동/동네/ 출발/출발지,"출발,출발지"
2,다산콜센터,대중교통 안내,B2033,고객,3,버스노선,,A,,,가산동 주민센터입니다.,,"가산동, 주민센터",가산동/동네/ 주민센터/공공기관,"주민센터,공공기관"
3,다산콜센터,대중교통 안내,B2033,상담사,4,,버스노선,A,,,,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.,"가산동, 주민센터, 남대문시장,버스, 노선",가산동/동네/ 주민센터/공공기관/ 남대문시장/지명/ 버스/교통수단,"주민센터,교통수단"
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,,정류장,,정류장
5,다산콜센터,대중교통 안내,B2033,상담사,6,,버스정류장,A,,,,문성초등학교 정류장에서 탑승하시면 됩니다.,"문성초등학교, 정류장",문성초등학교/공공기관,"정류장,공공기관"


In [254]:
df_all.columns

Index(['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명 ', '용어사전', '지식베이스'],
      dtype='object')

In [263]:
df_clean = df_all.drop(columns=['도메인',  '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '상담사질문(요청)', '고객답변','개체명 ', '용어사전', '지식베이스'])

In [264]:
df_clean.sort_index()

,카테고리,고객질문(요청),상담사답변
0,대중교통 안내,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,
1,대중교통 안내,,
2,대중교통 안내,,
3,대중교통 안내,,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.
4,대중교통 안내,어느정류장에서 타야합니까?,
...,...,...,...
89297,일반행정 문의,,"입주신청서와 추천서, 급여내역서 외 기타 서류를 갖고 홈페이지에서 신청하면 됩니다."
89298,일반행정 문의,입주순위도 있나요?,
89299,일반행정 문의,,1~3순위가 있습니다.
89300,일반행정 문의,1순위는 누가 되나요?,


In [265]:
df_clean = df_clean[(df_clean['고객질문(요청)'] != "") | (df_clean['상담사답변'] != "")]

In [266]:
df_clean.value_counts()

카테고리     고객질문(요청)  상담사답변                                       
일반행정 문의            네                                               224
대중교통 안내            네                                               174
                   네, 그렇습니다.                                       117
                   네 그렇습니다.                                         97
일반행정 문의            네. 그렇습니다.                                        96
                                                                  ... 
대중교통 안내             20분정도 소요됩니다.                                     1
                    2000원입니다                                         1
                    17900원입니다.                                       1
                     어플을 이용하면 그러면 지금 서 계신곳을 중심으로 근처 버스정보가 나옵니다.      1
                                                                     1
Name: count, Length: 46028, dtype: int64

In [267]:
df_clean.head(10)

,카테고리,고객질문(요청),상담사답변
0,대중교통 안내,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,
3,대중교통 안내,,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.
4,대중교통 안내,어느정류장에서 타야합니까?,
5,대중교통 안내,,문성초등학교 정류장에서 탑승하시면 됩니다.
6,대중교통 안내,버스요금은 얼마입니까?,
7,대중교통 안내,,1200원 입니다.
8,대중교통 안내,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,
9,대중교통 안내,,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다."
10,대중교통 안내,시간은 얼마정도 걸립니까?,
11,대중교통 안내,,약 1시간 10분정도 걸립니다.


In [268]:
df_clean['상담사답변'] = df_clean['상담사답변'].shift(-1)

In [269]:
df_clean

,카테고리,고객질문(요청),상담사답변
0,대중교통 안내,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.
3,대중교통 안내,,
4,대중교통 안내,어느정류장에서 타야합니까?,문성초등학교 정류장에서 탑승하시면 됩니다.
5,대중교통 안내,,
6,대중교통 안내,버스요금은 얼마입니까?,1200원 입니다.
...,...,...,...
89297,일반행정 문의,,
89298,일반행정 문의,입주순위도 있나요?,1~3순위가 있습니다.
89299,일반행정 문의,,
89300,일반행정 문의,1순위는 누가 되나요?,국가유공자 자녀 및 국민기초생활수급권자를 1순위로 두고있습니다.


In [270]:
df_clean = df_clean.drop_duplicates(subset=['고객질문(요청)']).reset_index()
# df_clean[df_clean['label'] != 3]
seq_index = df_clean.index.astype(str)
df_clean.drop(columns=['index'], inplace=True)

In [272]:
# df_clean.drop(columns=['level_0'], inplace=True)

In [273]:
df_clean = df_clean.drop(1)

In [274]:
df_clean.reset_index(inplace=True)

In [275]:
df_clean.drop(columns=['index'], inplace=True)

In [276]:
df_clean.duplicated().sum()

np.int64(0)

In [277]:
df_clean.isna().sum()

카테고리        0
고객질문(요청)    0
상담사답변       0
dtype: int64

In [278]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22823 entries, 0 to 22822
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   카테고리      22823 non-null  object
 1   고객질문(요청)  22823 non-null  object
 2   상담사답변     22823 non-null  object
dtypes: object(3)
memory usage: 535.0+ KB


In [279]:
df_clean = df_clean[df_clean['고객질문(요청)'].astype(str).str.strip() != ""].reset_index(drop=True)

In [302]:
df_clean.isna().sum()

카테고리        0
고객질문(요청)    0
상담사답변       0
dtype: int64

In [280]:
df_clean.head(5)

,카테고리,고객질문(요청),상담사답변
0,대중교통 안내,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.
1,대중교통 안내,어느정류장에서 타야합니까?,문성초등학교 정류장에서 탑승하시면 됩니다.
2,대중교통 안내,버스요금은 얼마입니까?,1200원 입니다.
3,대중교통 안내,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다."
4,대중교통 안내,시간은 얼마정도 걸립니까?,약 1시간 10분정도 걸립니다.


In [281]:
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer  # 벡터화
from sklearn.metrics.pairwise import cosine_similarity

In [282]:
# 토큰화 -> 벡터화
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=tokenize,
    ngram_range=(1,2)
)

In [283]:
X = vectorizer.fit_transform(df_clean['고객질문(요청)'].astype(str).tolist())

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [284]:
# 질문 목록
new_questions = [
    '여권 재발급 신청 방법을 알려줘',
    '전입 신고가 인터넷으로 가능한가요?',
    '지방세 환급을 어디서 하나요?'
]

In [285]:
new_questions_vec = vectorizer.transform([question for question in new_questions])

In [286]:
new_questions_vec

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 50 stored elements and shape (3, 49878)>

In [287]:
# 코사인 거리 유사도, 함수를 사용
# ravel() -> array에서 사용하는 함수로 다차원 배열을 1차원 배열로 변경하는 함수
sims = cosine_similarity(new_questions_vec, X).ravel()

In [288]:
sims

array([0.01312473, 0.        , 0.        , ..., 0.0342113 , 0.02337648,
       0.02061893], shape=(68460,))

In [289]:
rank = sims.argsort()[::-1]

In [290]:
# ...existing code...
# 잘못된 출력 루프 대체 코드
for idx, q in enumerate(new_questions):
    q_vec = new_questions_vec[idx]                 # 각 질문의 벡터(1 x n_features)
    sims = cosine_similarity(q_vec, X).ravel()    # 코사인 유사도 계산
    rank = sims.argsort()[::-1]

    print("질문 :", q)
    for i in rank[:2]:
        print(f"index: {i}, 유사도: {round(sims[i],3)}")
        print("유사 질문 :", df_clean.iloc[i]['고객질문(요청)'])
        print("답변     :", df_clean.iloc[i]['상담사답변'])
    print()
# ...existing code...

질문 : 여권 재발급 신청 방법을 알려줘
index: 10501, 유사도: 0.604
유사 질문 : 신청방법을 알려주세요.
답변     : 주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
index: 13452, 유사도: 0.415
유사 질문 : 여권 재발급 받고 싶어요
답변     : 여권사진, 신분증 준비하시면 됩니다.

질문 : 전입 신고가 인터넷으로 가능한가요?
index: 14977, 유사도: 0.661
유사 질문 : 인터넷으로 가능한가요?
답변     : 인터넷으로 신청 가능합니다.
index: 20128, 유사도: 0.492
유사 질문 : 납부는 인터넷으로 가능한가요?
답변     : 네. 가능합니다.

질문 : 지방세 환급을 어디서 하나요?
index: 10987, 유사도: 0.573
유사 질문 : 신청을 어디서 하나요?
답변     : 주민등록상 거주지가 속한 시, 군또는 자기추내의 읍, 면, 동 주민센터에서 발급 가능합니다. 
index: 1198, 유사도: 0.545
유사 질문 : 예약을 어디서 하나요?
답변     : 코레일 어플이나 철도청 홈페이지에서 예약 가능합니다. 



### 문제 2
- 고객 질문 데이터들을 카테고리를 분류하는 모델을 생성
    - 고객질문 데이터들을 이용하여 토큰화, 벡터화 작업 (독립 변수)
    - 카테고리 일반행정, 대중교통을 타겟 데이터(종속 변수)
        - 카테고리 데이터를 LabelEncoder()를 이용하여 수치화 변환
    - SVC 모델을 이용하여 벡터화된 데이터와 카테고리 데이터를 이용하여 학습
    - new_questions 의 카테고리를 확인

In [291]:
df_all['카테고리'].unique()

array(['대중교통 안내', '일반행정 문의'], dtype=object)

In [292]:
df_all['카테고리'].value_counts()

카테고리
일반행정 문의    50336
대중교통 안내    38966
Name: count, dtype: int64

In [293]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22820 entries, 0 to 22819
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   카테고리      22820 non-null  object
 1   고객질문(요청)  22820 non-null  object
 2   상담사답변     22820 non-null  object
dtypes: object(3)
memory usage: 535.0+ KB


In [296]:
df_clean['카테고리'].value_counts()

카테고리
일반행정 문의    14058
대중교통 안내     8762
Name: count, dtype: int64

In [297]:
from sklearn.preprocessing import LabelEncoder

In [298]:
komoran = Komoran()
def tokenize(text):
    if not isinstance(text, str):
        return []
    s = text.strip()
    return komoran.morphs(s) if s else []

# 3) TF-IDF 벡터화 (vectorizer 새로 생성하여 fit)
vectorizer = TfidfVectorizer(lowercase=False, tokenizer=tokenize, ngram_range=(1,2))
X = vectorizer.fit_transform(df_clean['고객질문(요청)'].astype(str).tolist())

# 4) 레이블(카테고리) 인코딩
le = LabelEncoder()
y = le.fit_transform(df_clean['카테고리'].astype(str).tolist())

# 5) 확인 출력
print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("라벨 매핑:", dict(zip(le.classes_, le.transform(le.classes_))))

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


X.shape: (22820, 49878)
y.shape: (22820,)
라벨 매핑: {np.str_('대중교통 안내'): np.int64(0), np.str_('일반행정 문의'): np.int64(1)}


In [299]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# 1) 학습/검증 분할 (클래스 불균형 고려해 stratify 사용)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2) SVC 생성 및 학습 (linear kernel 권장: 해석/속도)
clf = SVC(kernel='linear', probability=True, random_state=42)
clf.fit(X_train, y_train)

# 3) 평가
y_pred = clf.predict(X_test)
print("accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 4) (선택) 혼동행렬 출력
print("confusion matrix:")
print(confusion_matrix(y_test, y_pred))

# 5) new_questions 예측 예시 (vectorizer, le 사용)
try:
    new_questions  # 존재 확인
except NameError:
    new_questions = [
        '여권 재발급 신청 방법을 알려줘',
        '전입 신고가 인터넷으로 가능한가요?',
        '지방세 환급을 어디서 하나요?'
    ]

new_vec = vectorizer.transform([str(q) for q in new_questions])
pred = clf.predict(new_vec)
proba = clf.predict_proba(new_vec)

for q, p, prob in zip(new_questions, pred, proba):
    print("질문:", q)
    print("예측 카테고리:", le.inverse_transform([p])[0], "신뢰도:", round(prob.max(), 3))

accuracy: 0.8986
              precision    recall  f1-score   support

     대중교통 안내       0.93      0.79      0.86      1752
     일반행정 문의       0.88      0.96      0.92      2812

    accuracy                           0.90      4564
   macro avg       0.91      0.88      0.89      4564
weighted avg       0.90      0.90      0.90      4564

confusion matrix:
[[1389  363]
 [ 100 2712]]
질문: 여권 재발급 신청 방법을 알려줘
예측 카테고리: 일반행정 문의 신뢰도: 1.0
질문: 전입 신고가 인터넷으로 가능한가요?
예측 카테고리: 일반행정 문의 신뢰도: 0.988
질문: 지방세 환급을 어디서 하나요?
예측 카테고리: 일반행정 문의 신뢰도: 0.972
